# 05 · Validation plan — activity + metal-incorporation assay, controls, Co substitution

**Standard slot:** *validation plan.* **For Project 20 this means:** turn the metal-geometry-filtered
set into a costed **activity + metal-incorporation** assay plan with the right controls (incl. an
**apo** enzyme and a **natural CA**), and an **alternative-metal Co(II) substitution** test for hits
(D4/D5).

This is the deliverable that states, plainly: **in-silico geometry is a hypothesis; the assay tests
activity and ICP tests whether the metal even binds.**

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Select the synthesis set (< 96 designs)
Pick the top designs from the ranked filter (metal-geometry first), capped at **< 96** so they fit a
single screening plate with controls. Diversity matters — don't pick 96 near-identical designs.

In [ ]:
import pandas as pd
try:
    ranked = pd.read_csv("results/ranked.csv")
except FileNotFoundError:
    ranked = pd.read_csv("results/campaign.csv")

# Prefer designs passing the metal-geometry bar; cap under 96 (leave wells for controls).
ok = ranked[ranked["catalytic_geom_rmsd"] <= 0.5] if "catalytic_geom_rmsd" in ranked else ranked
selected = ok.head(84).copy()
selected.to_csv("results/synthesis_set.csv", index=False)
print(f"selected {len(selected)} designs for synthesis (< 96, leaving wells for controls) [SYNTHETIC ranking]")
print("Diversify across scaffolds; record why each was chosen in your report.")

## 2 · Metal incorporation FIRST — it gates everything
A design with no bound Zn cannot be a metalloenzyme, regardless of geometry. Check incorporation
**before** trusting any activity:
- **ICP-MS** — bound metal (Zn) per protein; the quantitative gold standard.
- **PAR / 4-(2-pyridylazo)resorcinol** colorimetric assay — release the metal with a chelator and read
  it colorimetrically; a quick bench check of Zn stoichiometry.

Express in *E. coli* BL21(DE3), 16–18 °C overnight; His-tag → IMAC → SEC; **supplement Zn(II)** in the
medium/buffer to favour loading. Report **Zn : protein stoichiometry** — aim near 1:1.

## 3 · The activity assays (esterase proxy + true CO₂ hydration)
- **Esterase proxy (fast bench readout):** **p-nitrophenyl acetate (pNPA)** hydrolysis → follow
  **p-nitrophenolate** absorbance (~348–405 nm) in a plate reader; initial rates across substrate
  concentrations. Convenient chromogenic kinetics (CA has a promiscuous esterase activity).
- **True CO₂ hydration:** the classic **Wilbur-Anderson** assay — time the **pH drop** as CO₂ is
  hydrated; report **WA units** (stopped-flow for fast designs).
- **Note:** the esterase proxy and CO₂-hydration activity are **correlated but not identical** — state
  which you measured. Subtract the uncatalysed background (both reactions proceed without enzyme).

In [ ]:
controls = {
    "POSITIVE — natural carbonic anhydrase": "a verified CA (e.g. bovine/human CA II); confirms the assay works",
    "NEGATIVE — apo enzyme (metal stripped)": "SAME design, Zn removed with a chelator; the metalloenzyme "
        "analogue of a dead mutant — loss of activity pins catalysis to the metal",
    "NEGATIVE — His -> Ala metal-knockout mutant": "SAME design, a His ligand mutated to Ala; cannot bind the metal",
    "NEGATIVE — empty-vector lysate": "no insert; rules out host-background activity",
    "BLANK — buffer + substrate only": "the uncatalysed CO2-hydration / pNPA background rate to subtract",
}
print("MANDATORY controls (every plate):")
for k, v in controls.items():
    print(f"  - {k}\n      {v}")
print("\nMETAL-INCORPORATION gate: run ICP-MS / PAR on every candidate BEFORE trusting its activity.")

## 4 · A costed, plate-based screen (template — fill real prices)
One 96-well plate holds the < 96 designs + the controls above. Cost the gene synthesis, expression,
purification, the **ICP metal check**, and assay reagents at your institution's rates.

In [ ]:
plan = [
    ("Gene synthesis (codon-optimised, screened provider)", "< 96 designs", "fill price/construct"),
    ("Cloning + transformation", "1 plate", "fill"),
    ("Expression + lysis (Zn-supplemented)", "1 plate", "fill"),
    ("IMAC + SEC purification (plate format)", "1 plate", "fill"),
    ("ICP-MS / PAR metal-incorporation check", "per candidate", "fill"),
    ("p-nitrophenyl acetate (pNPA) substrate", "stock", "fill"),
    ("Wilbur-Anderson CO2 assay (gas + pH stack)", "per design", "fill"),
    ("Plate-reader / stopped-flow time (kinetics)", "per plate", "fill"),
]
print("Costed reagent/step list (fill institutional prices) [TEMPLATE]:")
for step, scale, cost in plan:
    print(f"  - {step:52s} {scale:14s} {cost}")
print("\nTimeline (typical): synthesis 2-3 wk -> clone/express 1-2 wk -> purify+ICP+assay 2-3 wk.")
print("Synthesis MUST go through an IGSC-member, biosecurity-screening provider (low dual-use here, "
      "but it is policy). Wet-lab needs institutional biosafety/ethics sign-off.")

## 5 · Alternative metal — Co(II) substitution `[stretch]`
Zn(II) is **spectroscopically silent**; **Co(II)** is active in carbonic anhydrase **and** gives a
diagnostic UV-vis d-d band. So a Co-substituted version is a powerful orthogonal confirmation that the
designed site is a **genuine metal site**:
- **Reconstitute:** strip to apo (chelator), then add Co(II); confirm uptake by ICP.
- **Spectroscopy:** look for the Co(II) d-d absorption band (a CA-like signature).
- **Activity:** measure pNPA / CO₂ activity with Co(II) vs Zn(II) vs apo.
A Co-restored activity + the expected band, with the apo form dead, is strong evidence the site is
real — much more than geometry alone.

In [ ]:
print("Co(II)-substitution loop (stretch): apo (strip Zn) -> add Co(II) -> ICP confirms uptake ->")
print("UV-vis for the Co(II) d-d band -> pNPA/CO2 activity (Co vs Zn vs apo).")
print("Reminder for the thesis: report Zn:protein stoichiometry, the hit rate, and that geometry,")
print("metal incorporation, and activity are THREE separate claims — each needs its own evidence.")

## D4 / D5 checklist
- [ ] `results/synthesis_set.csv`: < 96 diverse, metal-geometry-passing designs.
- [ ] **Metal-incorporation** plan (ICP-MS / PAR) run BEFORE activity — report Zn:protein stoichiometry.
- [ ] Activity-assay plan: pNPA esterase proxy and/or Wilbur-Anderson CO₂ units, with **all** controls
      (apo, His→Ala knockout, natural CA, empty vector, blank), costed + timed.
- [ ] Alternative-metal **Co(II) substitution** test for hits `[stretch]`.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release; "geometry ≠ incorporation ≠ activity" + the metal-FF caveat stated plainly.

You're done — this is the **enzyme-family template** (Project 18) specialised to a **metal active site**.